# Feature Engineering — Capa 1: Datos SRI

**Objetivo:** Construir las 8 features base del modelo de clustering a partir del cruce entre el Golden Record manual `match_final_empresas_verificado.csv` y los archivos SRI provinciales.

Este notebook marca el inicio del flujo operativo del modelo final. A diferencia del carril automatizado de *Entity Resolution* documentado en `02_data_cleaning`, aquí ya no se usan candidatos inferidos por similitud textual. La llave de integración es el RUC verificado manualmente, porque un error de identificación legal contaminaría las variables tributarias y, por extensión, los clusters obtenidos por K-Means.

## Flujo de este notebook
1. Cargar `match_final_empresas_verificado.csv` como fuente de verdad manual
2. Consolidar los archivos SRI provinciales en un solo DataFrame
3. Deduplicar el Golden Record a nivel RUC para evitar duplicar empresas por alias comercial
4. JOIN izquierdo: Golden Record ← SRI por RUC
5. Construir las 8 features (binarias, numéricas, categóricas)
6. Exportar `features_capa1.csv`

> **Nota:** `es_cliente_fpa` viaja como columna de referencia pero **NO se usa** en el modelo de clustering. Es solo para validación externa posterior.

> **Alcance vigente:** entran todos los RUC ecuatorianos verificados manualmente en `leads_ruc_new.xlsx` y `proyectos_empresa_ruc_new.xlsx`, aunque el pais comercial/origen del lead no sea Ecuador. De `proyectos_empresa_ruc_new.xlsx` nace `es_cliente_fpa = 1` porque refleja proyectos/horas reales de FPA.

> **Decisión metodológica:** el pipeline automatizado de Entity Resolution se conserva como baseline experimental y propuesta futura de productización. Este notebook usa exclusivamente el diccionario manual verificado para proteger la validez del enriquecimiento SRI.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import glob
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def find_project_root(start: Path) -> Path:
    """Permite ejecutar el notebook desde la raiz o desde una subcarpeta."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        expected = candidate / '02_data_cleaning/data_ruc_universo_empresas/match_final_empresas_verificado.csv'
        if expected.exists():
            return candidate
    raise FileNotFoundError('No se encontro match_final_empresas_verificado.csv en el proyecto')


# Rutas base
ROOT        = find_project_root(Path.cwd())
PATH_MATCH  = ROOT / '02_data_cleaning/data_ruc_universo_empresas/match_final_empresas_verificado.csv'
PATH_SRI    = ROOT / '02_data_cleaning/data_SRI'
PATH_OUTPUT = ROOT / '03_feature_engineering/outputs'
PATH_OUTPUT.mkdir(parents=True, exist_ok=True)

print('Rutas configuradas correctamente.')

Rutas configuradas correctamente.


## 1. Cargar Golden Record manual

Se ingiere el diccionario manual de alias construido a partir de la búsqueda verificada de RUCs. Este archivo resuelve la brecha estructural del CRM: los sistemas internos registran nombres comerciales, mientras que SRI y SCVS operan con RUC y razón social oficial. La versión vigente usa todos los RUC ecuatorianos verificados disponibles, independientemente del pais comercial/origen registrado en el CRM manual.

La unidad de observación se consolida a nivel de **RUC único**. Si varios alias comerciales apuntan al mismo RUC, se mantienen como trazabilidad, pero no duplican la observación del modelo.

In [2]:
# Cargamos el Golden Record manual: alias comercial -> razon social oficial -> RUC.
golden = pd.read_csv(PATH_MATCH, dtype={'ruc': str})

required_cols = {'nombre_original', 'nombre_original_norm', 'razon_social', 'razon_social_norm', 'ruc', 'fuentes'}
missing_cols = required_cols - set(golden.columns)
if missing_cols:
    raise ValueError(f'Faltan columnas requeridas en el Golden Record: {sorted(missing_cols)}')

# Limpiar RUC: eliminar caracteres no numericos y asegurar 13 digitos.
golden['RUC'] = (
    golden['ruc']
    .astype(str)
    .str.replace(r'\D+', '', regex=True)
    .str.zfill(13)
)
golden = golden[golden['RUC'].str.fullmatch(r'\d{13}')].copy()

# Trazabilidad de origen manual.
# proyectos_empresa_ruc_new.xlsx proviene de empresas con proyectos/horas FPA.
# leads_ruc_new.xlsx proviene de prospectos CRM.
fuentes_str = golden['fuentes'].astype('string').fillna('')
golden['has_leads'] = fuentes_str.str.contains('leads_ruc_new', case=False, regex=False)
golden['has_proyectos'] = fuentes_str.str.contains('proyectos_empresa_ruc_new', case=False, regex=False)


def join_unique(values):
    uniques = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in uniques:
            uniques.append(text)
    return ' | '.join(sorted(uniques))


# Unidad de observacion para clustering: empresa legal unica por RUC.
# Si un mismo RUC aparece como lead y cliente, se marca como cliente FPA.
golden = golden.sort_values(['RUC', 'has_proyectos', 'nombre_original_norm'], ascending=[True, False, True])
match = (
    golden.groupby('RUC', as_index=False)
    .agg(
        name_raw=('razon_social', 'first'),
        name_norm=('razon_social_norm', 'first'),
        alias_manual=('nombre_original', join_unique),
        n_aliases=('nombre_original_norm', 'nunique'),
        has_leads=('has_leads', 'max'),
        has_proyectos=('has_proyectos', 'max'),
    )
)

match['source_label'] = np.select(
    [match['has_proyectos'] & match['has_leads'], match['has_proyectos'], match['has_leads']],
    ['HORAS+LEADS', 'HORAS', 'LEADS'],
    default='DESCONOCIDO'
)
match['source_winner'] = 'GROUND_TRUTH_MANUAL'
match['score'] = 100
match['verdict'] = 'VERIFICADO_MANUAL'
match['es_cliente_fpa'] = match['has_proyectos'].astype(int)

assert match['RUC'].duplicated().sum() == 0, 'Hay RUCs duplicados despues de consolidar a nivel empresa legal'

print(f'Alias verificados en Golden Record: {len(golden)}')
print(f'Empresas legales unicas por RUC:    {len(match)}')
print(f'RUCs con mas de un alias CRM:       {(match["n_aliases"] > 1).sum()}')
print(f'Clientes FPA en dataset:            {match["es_cliente_fpa"].sum()} / {len(match)}')
print(f'Longitud RUC (debe ser todo 13):    {match["RUC"].str.len().value_counts().to_dict()}')
print(f'Distribucion source_label:          {match["source_label"].value_counts().to_dict()}')
print()
print(match[['name_norm', 'RUC', 'source_label', 'n_aliases', 'es_cliente_fpa']].head(8).to_string())

Alias verificados en Golden Record: 209
Empresas legales unicas por RUC:    181
RUCs con mas de un alias CRM:       16
Clientes FPA en dataset:            82 / 181
Longitud RUC (debe ser todo 13):    {13: 181}
Distribucion source_label:          {'LEADS': 99, 'HORAS': 77, 'HORAS+LEADS': 5}

                                                    name_norm            RUC source_label  n_aliases  es_cliente_fpa
0                                        BANCO DEL AUSTRO S A  0190055965001        HORAS          1               1
1                                        INDURAMA ECUADOR S A  0190061264001        HORAS          1               1
2                                            INDUATENAS S A S  0190087719001        LEADS          1               0
3                                            GRAIMAN CIA LTDA  0190122271001        HORAS          1               1
4                                     ASEGURADORA DEL SUR C A  0190123626001        HORAS          1               1
5     

## 2. Consolidar archivos SRI provinciales

Los archivos SRI tienen:
- Separador: `|`
- Encoding: `utf-8-sig`
- Columna RUC: `NUMERO_RUC`

Se consolidan todos y se deduplica por RUC (`keep='first'`) porque una empresa
puede tener establecimientos registrados en más de una provincia.

In [3]:
# Cargar y consolidar todos los archivos SRI provinciales
sri_files = glob.glob(str(PATH_SRI / 'SRI_RUC_*.csv'))
print(f'Archivos SRI encontrados: {len(sri_files)}')

chunks = []
for f in sri_files:
    try:
        df_chunk = pd.read_csv(f, sep='|', encoding='utf-8-sig', dtype=str, low_memory=False)
        chunks.append(df_chunk)
    except Exception as e:
        print(f'  ERROR {f}: {e}')

sri = pd.concat(chunks, ignore_index=True)
sri.columns = [c.strip() for c in sri.columns]
sri['NUMERO_RUC'] = sri['NUMERO_RUC'].astype(str).str.strip()

# Deduplicar: si una empresa aparece en 2+ provincias, conservar la primera aparición
sri_dedup = sri.drop_duplicates(subset='NUMERO_RUC', keep='first').copy()

print(f'Total registros SRI consolidados:         {len(sri):,}')
print(f'Registros únicos por RUC (deduplicados):  {len(sri_dedup):,}')
print(f'Columnas: {list(sri_dedup.columns)}')

Archivos SRI encontrados: 24


Total registros SRI consolidados:         8,050,507
Registros únicos por RUC (deduplicados):  6,799,570
Columnas: ['NUMERO_RUC', 'RAZON_SOCIAL', 'CODIGO_JURISDICCION', 'ESTADO_CONTRIBUYENTE', 'CLASE_CONTRIBUYENTE', 'FECHA_INICIO_ACTIVIDADES', 'FECHA_ACTUALIZACION', 'FECHA_SUSPENSION_DEFINITIVA', 'FECHA_REINICIO_ACTIVIDADES', 'OBLIGADO', 'TIPO_CONTRIBUYENTE', 'NUMERO_ESTABLECIMIENTO', 'NOMBRE_FANTASIA_COMERCIAL', 'ESTADO_ESTABLECIMIENTO', 'DESCRIPCION_PROVINCIA_EST', 'DESCRIPCION_CANTON_EST', 'DESCRIPCION_PARROQUIA_EST', 'CODIGO_CIIU', 'ACTIVIDAD_ECONOMICA', 'AGENTE_RETENCION', 'ESPECIAL']


## 3. JOIN Golden Record ← SRI por RUC

Se usa **LEFT JOIN** para conservar las empresas legales verificadas aunque alguna no se encuentre
en el catálogo SRI. Las empresas sin match quedarán con `NaN` en las columnas SRI
y serán excluidas del dataset final de features (documentadas como limitación).

Este cruce ya no depende de similitud de nombres. La integración se realiza con una llave oficial (`RUC` ↔ `NUMERO_RUC`), lo que reduce el riesgo de falsos positivos en la construcción de variables.

In [4]:
# JOIN izquierdo: Golden Record manual ← SRI por RUC
df = match.merge(
    sri_dedup,
    left_on='RUC',
    right_on='NUMERO_RUC',
    how='left'
)

# Diagnóstico del join
con_datos_sri = df['NUMERO_RUC'].notna().sum()
sin_datos_sri = df['NUMERO_RUC'].isna().sum()

print(f'Total empresas tras JOIN:  {len(df)}')
print(f'  Con datos SRI:           {con_datos_sri}')
print(f'  Sin datos SRI (excluir): {sin_datos_sri}')
print()

if sin_datos_sri > 0:
    print('Empresas SIN match en SRI (se excluirán del modelo):')
    print(df[df['NUMERO_RUC'].isna()][['name_norm', 'RUC', 'source_label', 'source_winner', 'es_cliente_fpa']].to_string())

Total empresas tras JOIN:  181
  Con datos SRI:           181
  Sin datos SRI (excluir): 0



## 4. Construcción de las 8 features

| # | Feature | Columna SRI origen | Transformación |
|---|---|---|---|
| 1 | `tipo_sociedad` | `TIPO_CONTRIBUYENTE` | SOCIEDAD=1, otro=0 |
| 2 | `obligado_contabilidad` | `OBLIGADO` | S=1, N=0 |
| 3 | `es_agente_retencion` | `AGENTE_RETENCION` | S=1, N=0 |
| 4 | `es_contribuyente_especial` | `ESPECIAL` | S=1, N=0 |
| 5 | `estado_activo` | `ESTADO_CONTRIBUYENTE` | ACTIVO=1, otro=0 |
| 6 | `antiguedad_anos` | `FECHA_INICIO_ACTIVIDADES` | 2026 − año(fecha) |
| 7 | `sector_ciiu_macro` | `CODIGO_CIIU` | Primera letra → G/C/M/K/S/OTRO |
| 8 | `region` | `DESCRIPCION_PROVINCIA_EST` | Pichincha/Guayas/Resto |

In [5]:
# ── Feature 1: tipo_sociedad ──────────────────────────────────────────────────
# SOCIEDAD = empresa jurídica formalmente constituida (SA, Ltda, etc.)
# PERSONA NATURAL = persona natural con RUC y actividad económica
df['tipo_sociedad'] = (df['TIPO_CONTRIBUYENTE'].str.strip().str.upper() == 'SOCIEDAD').astype(int)

# ── Feature 2: obligado_contabilidad ─────────────────────────────────────────
# Empresa obligada a llevar contabilidad formal.
# El SRI obliga a empresas con ingresos > 300K USD anuales o activos > 180K USD.
# Es un proxy indirecto de tamaño económico mínimo.
df['obligado_contabilidad'] = (df['OBLIGADO'].str.strip().str.upper() == 'S').astype(int)

# ── Feature 3: es_agente_retencion ───────────────────────────────────────────
# Autorizado por el SRI para retener impuestos en sus pagos a proveedores.
# Indica mayor volumen de transacciones y relaciones comerciales activas.
df['es_agente_retencion'] = (df['AGENTE_RETENCION'].str.strip().str.upper() == 'S').astype(int)

# ── Feature 4: es_contribuyente_especial ──────────────────────────────────────
# Designación SRI para empresas de gran tamaño fiscal (mayor control tributario).
# En Ecuador es un indicador de empresa grande o con alta relevancia económica.
df['es_contribuyente_especial'] = (df['ESPECIAL'].str.strip().str.upper() == 'S').astype(int)

# ── Feature 5: estado_activo ──────────────────────────────────────────────────
# Empresa activa (en operación) vs suspendida o pasiva.
# Relevante para cualificar si el lead es un prospecto válido.
df['estado_activo'] = (df['ESTADO_CONTRIBUYENTE'].str.strip().str.upper() == 'ACTIVO').astype(int)

# ── Feature 6: antiguedad_anos ────────────────────────────────────────────────
# Madurez del negocio: empresas más antiguas tienen estructuras más consolidadas
# y mayor propensión a invertir en proyectos tecnológicos.
# Formato real en SRI: 'YYYY-MM-DD HH:MM:SS' → se parsea sin formato explícito
df['FECHA_INICIO_ACTIVIDADES'] = pd.to_datetime(
    df['FECHA_INICIO_ACTIVIDADES'], errors='coerce'
)
df['antiguedad_anos'] = 2026 - df['FECHA_INICIO_ACTIVIDADES'].dt.year
# Clip: eliminar valores negativos (fechas futuras en el registro SRI = error de datos)
df['antiguedad_anos'] = df['antiguedad_anos'].clip(lower=0)

# ── Feature 7: sector_ciiu_macro ──────────────────────────────────────────────
# Primera letra del código CIIU = sector económico a nivel macro.
# G=Comercio, C=Manufactura, M=Profesional/Técnico, K=Financiero, S=Servicios, OTRO
SECTORES_PRINCIPALES = {'G', 'C', 'M', 'K', 'S'}
df['sector_ciiu_macro'] = (
    df['CODIGO_CIIU'].astype(str).str.strip().str[0].str.upper()
)
df['sector_ciiu_macro'] = df['sector_ciiu_macro'].where(
    df['sector_ciiu_macro'].isin(SECTORES_PRINCIPALES), other='OTRO'
)

# ── Feature 8: region ─────────────────────────────────────────────────────────
# Las dos provincias con mayor concentración de empresas FPA; el resto agrupado.
def asignar_region(prov):
    if pd.isna(prov):
        return None
    prov_upper = str(prov).strip().upper()
    if 'PICHINCHA' in prov_upper:
        return 'Pichincha'
    elif 'GUAYAS' in prov_upper:
        return 'Guayas'
    else:
        return 'Resto'

df['region'] = df['DESCRIPCION_PROVINCIA_EST'].apply(asignar_region)

# ── Resumen de distribuciones ─────────────────────────────────────────────────
print('=== Distribuciones de las 8 features ===')
print()
BINARIAS = ['tipo_sociedad', 'obligado_contabilidad', 'es_agente_retencion',
            'es_contribuyente_especial', 'estado_activo']
for col in BINARIAS:
    dist = df[col].value_counts(dropna=False).to_dict()
    print(f'  {col}: {dist}')
print()
antig = df['antiguedad_anos']
print(f'  antiguedad_anos: min={antig.min():.0f}, mediana={antig.median():.0f}, '
      f'max={antig.max():.0f}, nulos={antig.isna().sum()}')
print()
print(f'  sector_ciiu_macro: {df["sector_ciiu_macro"].value_counts(dropna=False).to_dict()}')
print(f'  region:            {df["region"].value_counts(dropna=False).to_dict()}')

=== Distribuciones de las 8 features ===

  tipo_sociedad: {1: 181}
  obligado_contabilidad: {1: 181}
  es_agente_retencion: {1: 142, 0: 39}
  es_contribuyente_especial: {1: 137, 0: 44}
  estado_activo: {1: 163, 0: 18}

  antiguedad_anos: min=2, mediana=31, max=105, nulos=0

  sector_ciiu_macro: {'G': 66, 'C': 42, 'OTRO': 39, 'K': 20, 'M': 14}
  region:            {'Pichincha': 78, 'Guayas': 56, 'Resto': 47}


## 5. Exportar features_capa1.csv

El archivo exportado contiene:
- **Columnas de identidad:** `name_norm`, `RUC`, `source_label`, `source_winner`
- **Label de validación:** `es_cliente_fpa` (NO entra al modelo de clustering)
- **8 features:** listas para encoding y escalado en el notebook `03_matriz_final.ipynb`

La salida queda a nivel de RUC único, por lo que los alias comerciales del CRM no duplican observaciones del modelo. Esta decisión evita que una misma empresa legal pese varias veces en el clustering solo por aparecer con distintos nombres comerciales.

In [6]:
COLS_ID       = ['name_norm', 'RUC', 'source_label', 'source_winner', 'es_cliente_fpa']
COLS_FEATURES = [
    'tipo_sociedad',
    'obligado_contabilidad',
    'es_agente_retencion',
    'es_contribuyente_especial',
    'estado_activo',
    'antiguedad_anos',
    'sector_ciiu_macro',
    'region'
]

# Solo empresas CON datos SRI (las sin match se excluyen del modelo)
features_capa1 = df[df['NUMERO_RUC'].notna()][COLS_ID + COLS_FEATURES].copy()
features_capa1 = features_capa1.reset_index(drop=True)

assert features_capa1['RUC'].duplicated().sum() == 0, 'features_capa1 debe quedar a nivel RUC unico'

# Diagnóstico de nulos en las features
print('=== Diagnóstico de nulos por feature ===')
for col in COLS_FEATURES:
    n_null = features_capa1[col].isna().sum()
    estado = 'OK' if n_null == 0 else f'{n_null} NULOS'
    print(f'  {col}: {estado}')

print()
print(f'Shape final features_capa1:        {features_capa1.shape}')
print(f'Clientes FPA en dataset:           {features_capa1["es_cliente_fpa"].sum()} / {len(features_capa1)}')
print(f'Distribución es_cliente_fpa:       {features_capa1["es_cliente_fpa"].value_counts().to_dict()}')
print(f'RUCs únicos en features_capa1:     {features_capa1["RUC"].nunique()}')

# Exportar
out_path = PATH_OUTPUT / 'features_capa1.csv'
features_capa1.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'\nExportado: {out_path}')

=== Diagnóstico de nulos por feature ===
  tipo_sociedad: OK
  obligado_contabilidad: OK
  es_agente_retencion: OK
  es_contribuyente_especial: OK
  estado_activo: OK
  antiguedad_anos: OK
  sector_ciiu_macro: OK
  region: OK

Shape final features_capa1:        (181, 13)
Clientes FPA en dataset:           82 / 181
Distribución es_cliente_fpa:       {0: 99, 1: 82}
RUCs únicos en features_capa1:     181

Exportado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\03_feature_engineering\outputs\features_capa1.csv
